# TD 6 · Mini-défis : diagnostiquer, concevoir, conclure

**Bloc 3 : deep learning · Séances 8 à 10 · Durée : 55 min · Niveau : ★★★ (autonomie)**

---

## Objectifs

Ce dernier TD est **autonome** : peu d'indications, des critères de réussite
explicites. À la fin, vous aurez :

1. trouvé et corrigé trois bugs silencieux dans un script qui « marche », et mesuré le
   coût de chacun ;
2. conçu et exécuté votre propre ablation, après en avoir rédigé la pre-registration,
   pour répondre à une question laissée ouverte par la séance 10 ;
3. rédigé une recommandation bornée, défendable devant un client et devant un jury.

## Prérequis

TD 1 à TD 5. Ce TD mobilise tout le bloc : training loop, checklist de débogage,
data leakage (séance 3), `model.eval()`, ablation avec seeds, comparaison honnête.

## Concepts mobilisés

checklist de débogage · data leakage · double sigmoïde · `model.eval()` · pre-registration
d'une ablation · seuil de décision · interactions entre régularisateurs · portée d'une
conclusion · reproductibilité (seeds, folds, nombre de threads).

## Déroulé

| Défi | Contenu                  | Durée  | Type                          | Niveau |
| ---- | ------------------------ | ------ | ----------------------------- | ------ |
| 1    | Le script qui « marche » | 20 min | déboguer, mesurer             | ★★★    |
| 2    | Votre ablation           | 20 min | concevoir, exécuter, conclure | ★★★    |
| 3    | La recommandation        | 15 min | rédiger, s'auto-évaluer       | ★★★    |

## Résultats attendus

- Défi 1 : trois bugs nommés, un script corrigé, et un tableau qui mesure le coût de
  chaque bug séparément.
- Défi 2 : une fiche de pre-registration remplie **avant** l'exécution, un tableau
  de résultats sur trois seeds, et une conclusion qui respecte votre propre règle.
- Défi 3 : un paragraphe de recommandation qui coche la grille d'auto-évaluation.

## Mode d'emploi

Travaillez en binôme. Pour chaque défi, les **critères de réussite** sont donnés au
début : vérifiez-les vous-mêmes avant d'appeler l'enseignant.


## Mise en route


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import re

import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host

set_seed(SEED)
torch.set_num_threads(4)

df = load_raw()
train, _ = split_by_host(df)
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False
)
NUMERIC = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "calculated_host_listings_count",
    "availability_365",
]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]
X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
train_idx, val_idx = list(GroupKFold(n_splits=5).split(X, y, groups=train["host_id"]))[
    0
]
y_tr, y_va = y.iloc[train_idx].to_numpy(), y.iloc[val_idx].to_numpy()


def make_preprocessor():
    return ColumnTransformer(
        [
            (
                "num",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler()),
                    ]
                ),
                NUMERIC,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                min_frequency=20,
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                CATEGORICAL,
            ),
        ]
    )


print(f"fold 0 : {len(train_idx)} lignes d'entraînement, {len(val_idx)} de validation")

---

## Défi 1 · Le script qui « marche »

_Déboguer, mesurer · 20 min · ★★★_

Un collègue vous transmet sa fonction d'entraînement. Elle s'exécute sans erreur et
renvoie une AUC d'apparence correcte. Elle contient pourtant **trois bugs
silencieux**.

**Critères de réussite :**

- les trois bugs sont nommés, avec l'indice (une mesure ou une lecture) qui les a
  trahis ;
- deux évaluations successives du **même** modèle corrigé donnent la même AUC ;
- un tableau donne, sur trois seeds, l'AUC du script d'origine, celle de chaque
  correction isolée, et celle du script entièrement corrigé.

**Outils autorisés :** la checklist du TD 4, le test des 20 lignes, et une idée
simple : **évaluer deux fois le même modèle entraîné**. (Relancer toute la fonction ne
suffit pas : les seeds rendent les deux exécutions identiques, bugs compris.)


In [ ]:
def entrainement_collegue(graine=0):
    # Préparation des données.
    lignes = np.r_[train_idx, val_idx]
    prep = make_preprocessor().fit(X.iloc[lignes])
    X_a, X_b = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[val_idx])
    tr_l, va_l = loaders_from_arrays(X_a, y_tr, X_b, y_va, batch_size=512, seed=graine)
    # Modèle et optimizer.
    torch.manual_seed(graine)
    net = make_mlp(X_a.shape[1], hidden=(64, 32), dropout=0.3)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    perte_fn = nn.BCEWithLogitsLoss()
    # Entraînement.
    for epoque in range(25):
        net.train()
        for xb, yb in tr_l:
            opt.zero_grad()
            perte = perte_fn(torch.sigmoid(net(xb)), yb)
            perte.backward()
            opt.step()
    # Évaluation.
    with torch.no_grad():
        scores = np.vstack([net(xb).numpy() for xb, _ in va_l])
    return roc_auc_score(y_va, scores.ravel())


print(f"  AUC du script du collègue : {entrainement_collegue(0):.4f}")

In [ ]:
# TODO :
#  1. Enquêtez (exécutez, mesurez, relisez) et remplissez `bugs`.
#  2. Écrivez entrainement_corrige(graine, corriger=("A", "B", "C"), evaluations=1) : la
#     même fonction, où chaque lettre de `corriger` active la correction d'un bug, et qui
#     évalue `evaluations` fois le modèle entraîné (liste d'AUC si evaluations > 1).
#  3. Construisez le tableau demandé dans les critères de réussite (seeds 0, 1, 2).
# bugs = {"A": "____", "B": "____", "C": "____"}

bugs = {
    "A": "data leakage : le préprocesseur est ajusté sur training set + "
    "validation set (séance 3) ; trahi par la relecture de `lignes`",
    "B": "double sigmoïde : BCEWithLogitsLoss reçoit des probabilités ; trahi par le "
    "test des 20 lignes (la loss plafonne)",
    "C": "pas de net.eval() à l'évaluation : le dropout reste actif ; trahi en évaluant "
    "deux fois le même modèle, qui ne donne pas la même AUC",
}


def entrainement_corrige(graine=0, corriger=("A", "B", "C"), evaluations=1):
    lignes = train_idx if "A" in corriger else np.r_[train_idx, val_idx]
    prep = make_preprocessor().fit(X.iloc[lignes])
    X_a, X_b = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[val_idx])
    tr_l, va_l = loaders_from_arrays(X_a, y_tr, X_b, y_va, batch_size=512, seed=graine)
    torch.manual_seed(graine)
    net = make_mlp(X_a.shape[1], hidden=(64, 32), dropout=0.3)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    perte_fn = nn.BCEWithLogitsLoss()
    for epoque in range(25):
        net.train()
        for xb, yb in tr_l:
            opt.zero_grad()
            sortie = net(xb) if "B" in corriger else torch.sigmoid(net(xb))
            perte = perte_fn(sortie, yb)
            perte.backward()
            opt.step()
    if "C" in corriger:
        net.eval()
    aucs = []
    with torch.no_grad():
        for _ in range(evaluations):
            scores = np.vstack([net(xb).numpy() for xb, _ in va_l])
            aucs.append(roc_auc_score(y_va, scores.ravel()))
    return aucs[0] if evaluations == 1 else aucs


for nom, texte in bugs.items():
    print(f"  bug {nom} : {texte}")

In [ ]:
# L'indice le plus simple : évaluer deux fois le même modèle entraîné.
if "entrainement_corrige" in globals():
    print(
        "  deux évaluations du même modèle, script d'origine :",
        np.round(entrainement_corrige(0, corriger=(), evaluations=2), 4),
    )
    print(
        "  deux évaluations du même modèle, script corrigé   :",
        np.round(entrainement_corrige(0, evaluations=2), 4),
    )
    # Le coût de chaque bug, mesuré comme un traitement de la séance 10 : trois seeds.
    versions = {
        "script d'origine": (),
        "correction A seule (data leakage)": ("A",),
        "correction B seule (double sigmoïde)": ("B",),
        "correction C seule (model.eval)": ("C",),
        "tout corrigé": ("A", "B", "C"),
    }
    lignes = []
    for nom, corr in versions.items():
        a = np.array([entrainement_corrige(g, corriger=corr) for g in (0, 1, 2)])
        lignes.append(
            {
                "version": nom,
                "AUC": round(a.mean(), 4),
                "sd": round(a.std(ddof=1), 4),
                "par seed": np.round(a, 4).tolist(),
            }
        )
    tableau = pd.DataFrame(lignes)
    print()
    print(tableau.to_string(index=False))
else:
    print("  à compléter")

**Pour analyser votre tableau.** Un bug peut coûter beaucoup (il se voit dans l'AUC)
ou presque rien sur ce fold (il ne se voit pas, mais il invalide la méthode). Les deux
doivent être corrigés : le second parce qu'il fausserait toute évaluation sur des
données dont la distribution diffère davantage.

**Question D1.** Lequel des trois bugs aurait été le plus grave pour une **mise en
production**, et pourquoi n'est-ce pas forcément celui qui coûte le plus d'AUC ici ?


---

## Défi 2 · Votre ablation

_Concevoir, exécuter, conclure · 20 min · ★★★_

La séance 10 a trouvé que « tout combiné » (0,8532) ne bat pas le weight decay seul
(0,8547), et a proposé deux explications : la présence de la batch norm, qui n'avait
pas mérité sa place, et le fait que les régularisateurs ne s'additionnent pas. **Votre
question :** la batch norm est-elle responsable ?

**Critères de réussite :**

- la fiche de pre-registration est remplie **avant** toute exécution ;
- un seul facteur change entre les configurations comparées ;
- trois seeds par configuration, et la règle appliquée telle qu'écrite ;
- une conclusion qui énonce sa portée.

La fonction d'entraînement de la séance 10 est fournie.


In [ ]:
BIG = (256, 128, 64)
prep = make_preprocessor().fit(X.iloc[train_idx])
X_tr, X_va = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[val_idx])


def train_once(
    seed,
    *,
    dropout=0.0,
    weight_decay=0.0,
    batch_norm=False,
    early_stop=False,
    epochs=40,
    hidden=BIG,
    batch_size=512,
    lr=1e-3,
):
    tr_loader, va_loader = loaders_from_arrays(
        X_tr, y_tr, X_va, y_va, batch_size=batch_size, seed=seed
    )
    torch.manual_seed(seed)
    net = make_mlp(X_tr.shape[1], hidden=hidden, dropout=dropout, batch_norm=batch_norm)
    stopper = EarlyStopping(patience=5) if early_stop else None
    history = fit(
        net,
        tr_loader,
        va_loader,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        early_stopping=stopper,
        seed=seed,
    )
    _, logits = evaluate(net, va_loader, nn.BCEWithLogitsLoss())
    return roc_auc_score(y_va, logits.ravel()), len(history["train"])

In [ ]:
# TODO (1) : la fiche de pre-registration, AVANT d'exécuter quoi que ce soit.
# fiche = {
#     "hypothèse": "____",
#     "configurations comparées": "____",
#     "le seul facteur qui change": "____",
#     "seeds": "____",
#     "règle de décision (fixée maintenant)": "____",
#     "ce que je conclurai si la règle dit oui / si elle dit non": "____",
# }

# TODO (2) : exécutez les configurations (3 seeds chacune) et appliquez votre règle.

fiche = {
    "hypothèse": "retirer la batch norm de « tout combiné » améliore son AUC",
    "configurations comparées": "tout combiné (early stopping, wd 1e-3, dropout 0,2, batch norm) "
    "contre la même chose sans batch norm ; weight decay 1e-2 seul "
    "comme repère",
    "le seul facteur qui change": "la présence de la batch norm",
    "seeds": "0, 1, 2 (les mêmes pour chaque configuration : comparaison appariée par seed)",
    "règle de décision (fixée maintenant)": "effet réel si la différence moyenne dépasse deux fois "
    "le plus grand des deux écarts-types ET si les trois "
    "différences par seed ont le même signe",
    "ce que je conclurai si la règle dit oui / si elle dit non": "oui : la batch norm pénalisait la combinaison ; non : on ne peut pas l'affirmer",
}
for cle, valeur in fiche.items():
    print(f"  {cle} : {valeur}")

CONFIGS_D2 = {
    "tout combiné": dict(
        early_stop=True, weight_decay=1e-3, dropout=0.2, batch_norm=True
    ),
    "tout combiné sans batch norm": dict(
        early_stop=True, weight_decay=1e-3, dropout=0.2
    ),
    "weight decay 1e-2 seul": dict(weight_decay=1e-2),
}
debut = time.monotonic()
resultats = {
    nom: [train_once(g, **r) for g in (0, 1, 2)] for nom, r in CONFIGS_D2.items()
}
print(f"\n  ({time.monotonic() - debut:.0f} s)")
lignes = []
for nom, runs in resultats.items():
    a = np.array([r[0] for r in runs])
    lignes.append(
        {
            "configuration": nom,
            "AUC": round(a.mean(), 4),
            "sd": round(a.std(ddof=1), 4),
            "par seed": np.round(a, 4).tolist(),
            "epochs": [r[1] for r in runs],
        }
    )
d2 = pd.DataFrame(lignes)
print(d2.to_string(index=False))

avec = np.array([r[0] for r in resultats["tout combiné"]])
sans = np.array([r[0] for r in resultats["tout combiné sans batch norm"]])
diff = sans - avec
seuil = 2 * max(avec.std(ddof=1), sans.std(ddof=1))
meme_signe = bool(np.all(diff > 0) or np.all(diff < 0))
print(f"\n  sans - avec batch norm : {diff.mean():+.4f} (par seed {np.round(diff, 4)})")
print(f"  seuil : {seuil:.4f}   signes identiques : {meme_signe}")
print(
    f"  verdict de la règle : {'effet réel' if abs(diff.mean()) > seuil and meme_signe else 'non démontré'}"
)

**Question D2.** Votre règle a-t-elle tranché ? Écrivez la conclusion en deux phrases,
dont une qui dit ce que l'expérience **ne permet pas** d'affirmer.


---

## Défi 3 · La recommandation

_Rédiger, s'auto-évaluer · 15 min · ★★★_

Le client (la mairie de Barcelone, qui veut repérer les annonces probablement sans
licence) vous demande : « Faut-il utiliser un réseau de neurones ? » Rédigez la
section **Recommandation** de votre fiche de modèle (_Model Card_), en 120 à 180 mots,
à partir de **vos** résultats des TD 4, 5 et 6.

**Grille d'auto-évaluation** (cochez avant de rendre) :

| critère                                                                                  | oui / non |
| ---------------------------------------------------------------------------------------- | --------- |
| le modèle recommandé est nommé, avec un chiffre vérifiable (AUC, folds, test)            |           |
| l'incertitude est donnée (écart-type, signes par fold ou p-valeur)                       |           |
| la portée est énoncée (données tabulaires de Barcelone, ce protocole)                    |           |
| aucune affirmation plus forte que la preuve (ex. « la batch norm est mauvaise »)         |           |
| au moins une piste qui pourrait changer la conclusion (ex. le texte des annonces)        |           |
| les conditions de reproductibilité sont citées (seeds, folds groupés, nombre de threads) |           |


In [ ]:
# TODO : votre texte, entre les guillemets.
# recommandation = """
# ____
# """
# print(f"{len(recommandation.split())} mots")

recommandation = """
Nous recommandons HistGradientBoosting plutôt qu'un réseau de neurones pour ce problème.
Sur les annonces de Barcelone, avec cinq folds groupés par hôte et le même prétraitement,
le réseau (64-32) atteint une AUC de 0,9124 contre 0,9216 pour le booster : il perd dans
les cinq folds (paired t-test : p = 0,0062), et coûte plus de calcul à configuration égale.
Les réglages du réseau (batch size, learning rate, capacité) ne changent pas ce constat :
le réseau est sur un plateau. Sur le fold 0, même le meilleur grand réseau régularisé
(weight decay, AUC moyenne de 0,8547 sur trois seeds) reste sous le booster (0,8633).
Cette conclusion vaut pour ces features tabulaires et ce protocole. Elle pourrait changer
si l'on exploitait la description textuelle des annonces, qu'un arbre ne sait pas lire
(séance 11). Reproductibilité : seeds fixées, folds GroupKFold par hôte, torch limité à
4 threads ; un autre nombre de threads modifie les dernières décimales.
"""
print(recommandation.strip())
print(f"\n{len(recommandation.split())} mots")


"""
ce paragraphe de recommandation est un bon entraînement pour la soutenance de la séance 12 (portée, preuve, limites)
"""

---

## Synthèse du bloc (séances 8 à 10)

- **Représenter** : sans non-linéarité, un réseau est linéaire ; avec, il peut tout
  approcher sur un domaine borné, ce qui ne dit rien de l'apprentissage ni de la
  généralisation.
- **Apprendre** : la backpropagation est la chain rule avec réutilisation ;
  autograd la reproduit exactement.
- **Entraîner** : feature scaling dans le fold, suivi dès l'epoch 1, optimizer et
  learning rate ; lire les training curves avant de régler.
- **Généraliser** : mesurer chaque régularisateur, à plusieurs seeds, contre un seuil
  fixé à l'avance ; décider sur la métrique du problème.
- **Conclure** : un résultat est vrai **sur ce problème et dans ces conditions**, que
  l'on énonce.

## Pour aller plus loin

- Karpathy, A. (2019), _A Recipe for Training Neural Networks_.
  http://karpathy.github.io/2019/04/25/recipe/
- Mitchell, M. et al. (2019), _Model Cards for Model Reporting_, FAT\* 2019.
  https://arxiv.org/abs/1810.03993
- Sculley, D. et al. (2015), _Hidden Technical Debt in Machine Learning Systems_,
  NeurIPS (les bugs silencieux en production).
